### Unified errors
This notebook produces a dataframe with the attributes (Benchmark, prompt/problem_id, model, library, generated code, message errors, and error type) from the different benchmarks.
Models tested:
- gpt-4-turbo-2024-04-09
- gpt-4-0613
- gpt-3.5-turbo-0613
- gpt-3.5-turbo-0125
- claude-3-opus-20240229

In [1]:
import json
import re
import pandas as pd

In [2]:
def clean_error_msg(error_msg):
    """
        Retrieves error line and exception
    """
    if error_msg:
        text = 'line'
        pattern = re.compile(r'\b' + re.escape(text) + r'\b')
        lines = [match.start() for match in pattern.finditer(error_msg)]
        if len(lines) > 0:
            return error_msg[lines[-1]:]
    return ""

In [3]:
def read_jsonl(file_name, cols, unlist_cols, left_cols):
    """
        Returns a dataframe with chosen column names
        and columns to unlist
    """
    df_normalized = None
    normalized_dfs = []
    print(file_name)
    with open(file_name, 'r') as file:
        for line in file:
            data_line = json.loads(line)
            if len(unlist_cols) > 0:
                for column in unlist_cols:
                    normalized_column = pd.json_normalize(data_line[column])
                    for l in left_cols:
                        normalized_column[l] = data_line[l]
                    normalized_dfs.append(normalized_column)
                    # Concatenate the normalized DataFrames
                    df_normalized = pd.concat(normalized_dfs, axis=0, ignore_index=True)
            else:
                normalized_dfs.append(data_line)
                # Transform list to df
                df_normalized = pd.DataFrame(normalized_dfs)
    return df_normalized[cols]

In [4]:
def classify_error_msg(df):
    """
        [Remove function]
        Adds types of errors in the df
        given an error_message
    """
    # Tab error and Indentation error
    df['contains_indentation'] = df['message'].str.contains('indentation')
    # Syntax error
    df['contains_syntax'] = df['message'].str.contains('syntax')
    # Type Error
    df['contains_unexpected'] = df['message'].str.contains('unexpected')
    # Type or Value Error
    df['contains_must'] = df['message'].str.contains('must')
    # Attribute Error
    df['contains_has'] = df['message'].str.contains('has no')
    # Name Error
    # All elements in defined are in name (defined is a subelement of name)
    df['contains_name'] = df['message'].str.contains('name')
    df['contains_defined'] = df['message'].str.contains('defined')
    return df

In [36]:
def errors_type(error_msg):
    """
        Gets error type or exception description from error message
    """
    match = re.search(r'\b\w*Error\w*\b', error_msg)
    
    # If a match is found, return the matched word
    if match:
        return match.group(0)
    elif "Result not True" in error_msg:
        return "AssertionError"
    else:
        match = re.search(r'\b\w*re.error: .*$', error_msg)
        if match:
            return match.group(0)
        match = re.search(r'\b\w*Exception: .*$', error_msg)
        if match:
            return match.group(0)
        if error_msg == '' or error_msg == 'passed':
            return 'passed'
        return "Other error"

In [38]:
def get_experiments_by_path(paths=[], benchmark=''):
    """
        Function that unifies the error information of the different benchmarks.
        Returns a dataframe with these fields: benchmark, prompt/problem_id, model, 
        library, generated code, message errors, and error type
    """
    lst = []
    for path, model in paths:
        # specific processing for benchmark
        if benchmark == 'DS-1000':
            df = read_jsonl(path, cols=['result','library','perturbation_type','generate_code','completion_id'], unlist_cols=[], left_cols=['result','library','perturbation_type','generate_code','completion_id'])
            df["error_type"] = df["result"].apply(errors_type)
            df["model"] = [model for i in range(0, len(df))]
            df["benchmark"] = [benchmark for i in range(0, len(df))]
            df["prompt_id"] = [i+1 for i in range(0, 1000)]
            df.rename(columns={"result": "message", "completion_id":"_id"}, inplace=True)
        elif benchmark == 'CoderEval':
            df = read_jsonl(path, cols=['project','_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['project','_id'])
            df["message"] = df["eror_msg"].apply(clean_error_msg)
            df["error_type"] = df["message"].apply(errors_type)
            df["error_type"] = df["error_type"].replace('', 'passed')
            df["model"] = [model for i in range(0, len(df))]
            df["benchmark"] = [benchmark for i in range(0, len(df))]
            # Add prompt index ordering (10 samples of each prompt)
            prompt_id = []
            j = 0
            for i in range(0, len(df)):
                if i % 10 == 0:
                    j = j + 1
                prompt_id.append(j)
            df["prompt_id"] = prompt_id
            # Change nomenclature
            df.rename(columns={"project": "library"}, inplace=True)
        lst.append(df[['benchmark', 'model', 'prompt_id', '_id', 'library', 'generate_code', 'message', 'error_type']])
    return pd.concat(lst)

In [42]:
# CoderEval
paths = [('Data/CoderEval/CodeGen/Raw_Input.jsonl_out.jsonl', "350M CodeGen-Mono"), 
         ('Data/CoderEval/ChatGPT/Raw_Input.jsonl_out.jsonl', "gpt-3.5-turbo"), 
         ('Data/CoderEval/PanGu-Coder/Raw_Input.jsonl_out.jsonl', "300M PanGu-Coder")]
df_codereval = get_experiments_by_path(paths, benchmark='CoderEval')

Data/CoderEval/CodeGen/Raw_Input.jsonl_out.jsonl
Data/CoderEval/ChatGPT/Raw_Input.jsonl_out.jsonl
Data/CoderEval/PanGu-Coder/Raw_Input.jsonl_out.jsonl


In [40]:
# DS-1000
paths = [('Data/DS-1000/data/gpt-4-turbo-2024-04-09-0-errors.jsonl', 'gpt-4-turbo-2024-04-09'), 
         ('Data/DS-1000/data/gpt-4-0613-0-errors.jsonl', 'gpt-4-0613'), 
         ('Data/DS-1000/data/gpt-3.5-turbo-0613-0-errors.jsonl', 'gpt-3.5-turbo-0613'), 
         ('Data/DS-1000/data/gpt-3.5-turbo-0125-0-errors.jsonl', 'gpt-3.5-turbo-0125'),
         ('Data/DS-1000/data/claude-3-opus-20240229-0-errors.jsonl', 'claude-3-opus-20240229')]
df_ds1000 = get_experiments_by_path(paths, benchmark='DS-1000')

Data/DS-1000/data/gpt-4-turbo-2024-04-09-0-errors.jsonl
Data/DS-1000/data/gpt-4-0613-0-errors.jsonl
Data/DS-1000/data/gpt-3.5-turbo-0613-0-errors.jsonl
Data/DS-1000/data/gpt-3.5-turbo-0125-0-errors.jsonl
Data/DS-1000/data/claude-3-opus-20240229-0-errors.jsonl


In [53]:
len(df_codereval[df_codereval["error_type"]!="passed" ])

5916

In [54]:
len(df_ds1000[df_ds1000["error_type"]!="passed"])

2861

In [43]:
pd.concat([df_codereval, df_ds1000]).to_csv('unified_errors_0.csv', index=False)